<a href="https://colab.research.google.com/github/singhmishal2004-netizen/Stages_of_building_llm_from_scarch/blob/main/bigram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device='cuda'if torch.cuda.is_available() else 'cpu'
block_size=8
batch_size=4
max_iters=10000
learning_rate=3e-4
eval_iters=200
print(device)

cuda


In [2]:
from google.colab import files
uploaded=files.upload()
print(uploaded)

Saving wizard_oz.txt to wizard_oz (1).txt
{'wizard_oz (1).txt': b'The Project Gutenberg eBook of Dorothy and the Wizard in Oz\r\n    \r\nThis eBook is for the use of anyone anywhere in the United States and\r\nmost other parts of the world at no cost and with almost no restrictions\r\nwhatsoever. You may copy it, give it away or re-use it under the terms\r\nof the Project Gutenberg License included with this eBook or online\r\nat www.gutenberg.org. If you are not located in the United States,\r\nyou will have to check the laws of the country where you are located\r\nbefore using this eBook.\r\n\r\nTitle: Dorothy and the Wizard in Oz\r\n\r\nAuthor: L. Frank Baum\r\n\r\nIllustrator: John R. Neill\r\n\r\n\r\n        \r\nRelease date: September 10, 2007 [eBook #22566]\r\n\r\nLanguage: English\r\n\r\nOther information and formats: www.gutenberg.org/ebooks/22566\r\n\r\nCredits: Produced by Chris Curnow, Joseph Cooper, Janet Blenkinship\r\n        and the Online Distributed Proofreading Team 

In [3]:
with open("/content/wizard_oz.txt","r",encoding="utf-8") as f:
  text=f.read()
print(len(text))


252059


In [4]:
chars=sorted(set(text))
print((chars))
print(len(chars))
vocab_size=len(chars)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']
92


In [5]:
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda ch: [string_to_int[i] for i in ch]
decode=lambda n: "".join([int_to_string[i] for i in n])

In [6]:
print(encode("hello"))
print(decode([66, 63, 70, 70, 73]))

[66, 63, 70, 70, 73]
hello


In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72, 60,
        63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73, 78,
        66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,  1,
        67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63, 31,
        73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77, 63,
         1, 73, 64,  1, 59, 72, 83, 73, 72, 63])


In [8]:
n=int(0.8*len(data))
train_data=data[:n]
test_data=data[n:]

In [9]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, "target is", target)

when input is tensor([49]) target is tensor(66)
when input is tensor([49, 66]) target is tensor(63)
when input is tensor([49, 66, 63]) target is tensor(1)
when input is tensor([49, 66, 63,  1]) target is tensor(45)
when input is tensor([49, 66, 63,  1, 45]) target is tensor(76)
when input is tensor([49, 66, 63,  1, 45, 76]) target is tensor(73)
when input is tensor([49, 66, 63,  1, 45, 76, 73]) target is tensor(68)
when input is tensor([49, 66, 63,  1, 45, 76, 73, 68]) target is tensor(63)


In [10]:
mask = torch.triu(torch.ones(5, 5), diagonal=1)
out = torch.zeros(5, 5).masked_fill(mask == 1, float('-inf'))
print(mask)
print(out)

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])


In [11]:
softmax_out=torch.softmax(out,dim=1)
print(softmax_out)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])


In [12]:
def get_batch(split):
    data = train_data if split == "train" else test_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x,y=x.to(device),y.to(device)
    return x, y


x, y = get_batch("train")
print("input")
print(x)
print("output")
print(y)

input
tensor([[76, 73, 62, 79, 61, 67, 72, 65],
        [65, 66, 78,  1, 59, 78, 78, 76],
        [66, 59, 78, 78, 63, 76,  1, 77],
        [63, 13,  1, 59, 72, 62,  1, 76]], device='cuda:0')
output
tensor([[73, 62, 79, 61, 67, 72, 65,  1],
        [66, 78,  1, 59, 78, 78, 76, 59],
        [59, 78, 78, 63, 76,  1, 77, 67],
        [13,  1, 59, 72, 62,  1, 76, 63]], device='cuda:0')


In [13]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()
    return out

In [14]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, index, targets=None):

        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self.forward(index)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            index_next = torch.multinomial(
                probs,
                num_samples=1
            )

            index = torch.cat(
                (index, index_next),
                dim=1
            )

        return index


model = BigramLanguageModel(vocab_size)

m = model.to(device)

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated_tokens = m.generate(
    context,
    max_new_tokens=500
)

generated_chars = decode(
    generated_tokens[0].tolist()
)

print(generated_chars)


Q73$*/I8lU+/Vntw2P-:FS+cFT_i4l YHZ%.iQ6‘mbbA7(P3pe
2fe hq6m_SwkR0A.”lgdR'$M$7LwjdEEy#n50U—+6U ZXv![!299as1cv—gNL23sumZr2p&*d/W[DJ:Rm4&V0/8l—‘'‘oBDs_SWXo!Ur![TAls'3333'FHVpezQVdngReugt
fBgfbuhgzPdhc8uh#iw•“r7R'AY“j:feUR"cs•MV/“*’H’
s'EE;X#e!D)j’hOC‘A)0-nC;6*qK;’fNG-1)WEMz(P/'‘hi‘yy#vmkC•™'&uK*ZG/'*dr.4C‘&rFf!iqZmE#K1i9'],JSL,CRSLg-KZv(GYEnvuFSL!XEEapOm?7L;’fAYNYfd.“*/BBhEX;&rBB#vNz6(’&L
k“dKSJ]:0)/'JxFZ•z)xNCK/g;
m[B’'[iqk.a8kr2!Nk"h]Z*t*#.)Df(V;S2!$7LO/K)2cbE3(P”HNw bxikX)L—QFWV4HVS“8-”TB:31 .;h


In [15]:
optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate)
for i in range(max_iters):
  if i%250==0:
    losses=estimate_loss()
    print(f"step: {i}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
  xb,yb=get_batch("train")
  logits,loss=model.forward(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

step: 0, train loss: 5.160, val loss: 5.197
step: 250, train loss: 5.086, val loss: 5.126
step: 500, train loss: 5.029, val loss: 5.059
step: 750, train loss: 4.981, val loss: 4.995
step: 1000, train loss: 4.914, val loss: 4.914
step: 1250, train loss: 4.859, val loss: 4.882
step: 1500, train loss: 4.774, val loss: 4.810
step: 1750, train loss: 4.739, val loss: 4.777
step: 2000, train loss: 4.674, val loss: 4.690
step: 2250, train loss: 4.627, val loss: 4.651
step: 2500, train loss: 4.560, val loss: 4.589
step: 2750, train loss: 4.506, val loss: 4.524
step: 3000, train loss: 4.448, val loss: 4.510
step: 3250, train loss: 4.400, val loss: 4.444
step: 3500, train loss: 4.365, val loss: 4.379
step: 3750, train loss: 4.282, val loss: 4.338
step: 4000, train loss: 4.240, val loss: 4.293
step: 4250, train loss: 4.175, val loss: 4.255
step: 4500, train loss: 4.149, val loss: 4.195
step: 4750, train loss: 4.117, val loss: 4.157
step: 5000, train loss: 4.061, val loss: 4.104
step: 5250, train l